# Flipped-Class Notebook: Replaying Varian (2014) — *Big Data: New Tricks for Econometrics*

**Course:** Supervised ML for Business • **Today’s goals:** read, replicate, reflect.

**Paper:** Hal R. Varian (2014), *Journal of Economic Perspectives* 28(2):3–28.  
- Article link: https://www.aeaweb.org/articles?id=10.1257/jep.28.2.3  
- Replication data (OpenICPSR project 113925): https://www.openicpsr.org/openicpsr/project/113925  

**Group workflow:**
1) Skim the paper together (in class).  
2) Load one dataset from the replication package.  
3) Fit a baseline **OLS** model and one **ML model** (e.g., Random Forest).  
4) Compare predictions & discuss what ML added (or didn’t).  
5) Write 3–5 sentences reflecting on results & links to econometric thinking.

> You may use AI tools for coding help, but make sure you understand the code.


## 0) Setup
Run this cell to install and import what we need (works in Google Colab).

In [ ]:
# If running in Colab, uncomment the next two lines to ensure packages are present
# !pip -q install scikit-learn statsmodels

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
from pathlib import Path
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120
print('Ready! pandas', pd.__version__)

## 1) Get the data
**Option A (recommended):** Download the replication package from OpenICPSR (link above) and upload one CSV to this notebook (e.g., `FLS-data.csv` from the Lasso folder, or any other small CSV in the project).  

**Option B:** If you have a direct file URL, you can `wget` it into Colab.


In [ ]:
# ==== Option A: manual upload (Colab) ====
try:
    from google.colab import files  # type: ignore
    print("Choose a CSV from the replication package to upload (e.g., FLS-data.csv)")
    uploaded = files.upload()  # prompts dialog in Colab
    csv_name = list(uploaded.keys())[0]
except Exception as e:
    print('Upload helper not available (likely not running in Colab). You can set csv_name manually.')
    csv_name = None

# If you know the filename, set it explicitly here (uncomment):
# csv_name = 'FLS-data.csv'

print('csv_name =', csv_name)

In [ ]:
# ==== Option B: direct download via wget (if you have a direct URL) ====
# Example (replace URL with a direct CSV link if accessible without login):
# !wget -O FLS-data.csv "https://example.com/path/to/FLS-data.csv"
# csv_name = 'FLS-data.csv'

## 2) Inspect & clean
Load the CSV into a DataFrame, look at basic structure, and decide on **target (y)** and **features (X)**. Keep it simple (focus on a numeric target for regression).

In [ ]:
assert 'csv_name' in globals() and csv_name, "Set csv_name to your uploaded or downloaded CSV filename."
df = pd.read_csv(csv_name)
print(df.shape)
display(df.head())
display(df.describe(include='all').T)
print("Missing values per column:\n", df.isna().sum())

# TODO: pick target and features
target_col = None  # e.g., 'y' or a numeric column name
feature_cols = []  # e.g., ['x1','x2','x3']

# If there are categorical variables, one-hot encode them (example):
df_model = pd.get_dummies(df, drop_first=True)
df_model.head()

## 3) Train/test split
Create a holdout set to evaluate models fairly.

In [ ]:
assert target_col is not None, "Set target_col to your chosen target variable."
if not feature_cols:
    # If user didn't specify, use all numeric columns except target
    numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c != target_col]

X = df_model[feature_cols].copy()
y = df_model[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

## 4) Baseline model: OLS
Fit a simple linear regression as a benchmark using **statsmodels**.

In [ ]:
X_train_const = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_const).fit()
print(ols_model.summary())

X_test_const = sm.add_constant(X_test)
y_pred_ols = ols_model.predict(X_test_const)
ols_rmse = mean_squared_error(y_test, y_pred_ols, squared=False)
ols_mae = mean_absolute_error(y_test, y_pred_ols)
ols_r2 = r2_score(y_test, y_pred_ols)
print({"OLS_RMSE": ols_rmse, "OLS_MAE": ols_mae, "OLS_R2": ols_r2})

## 5) ML model: Random Forest
Fit a basic Random Forest regressor and compare performance. Tune a couple of hyperparameters quickly.

In [ ]:
rf = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [None, 6, 12],
    'min_samples_leaf': [1, 3]
}
gs = GridSearchCV(rf, param_grid, cv=3, n_jobs=-1, scoring='neg_root_mean_squared_error')
gs.fit(X_train, y_train)
print('Best params:', gs.best_params_)

best_rf = gs.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_rmse = mean_squared_error(y_test, y_pred_rf, squared=False)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)
print({"RF_RMSE": rf_rmse, "RF_MAE": rf_mae, "RF_R2": rf_r2})

# Quick feature importance plot (if available)
importances = getattr(best_rf, 'feature_importances_', None)
if importances is not None:
    fi = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(15)
    fi.plot(kind='bar')
    plt.title('Top feature importances (Random Forest)')
    plt.xlabel('Feature')
    plt.ylabel('Importance')
    plt.show()

## 6) Compare predictions
Visualize predicted vs. actual for OLS and RF.

In [ ]:
fig = plt.figure()
plt.scatter(y_test, y_pred_ols, alpha=0.6, label='OLS')
plt.scatter(y_test, y_pred_rf, alpha=0.6, label='RF')
miny, maxy = float(y_test.min()), float(y_test.max())
plt.plot([miny, maxy], [miny, maxy], linestyle='--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.title('Actual vs. Predicted (Test Set)')
plt.show()

pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R^2'],
    'OLS': [ols_rmse, ols_mae, ols_r2],
    'RandomForest': [rf_rmse, rf_mae, rf_r2]
})

## 7) (Optional) Counterfactual-style prediction
If your dataset has a clear **pre / post** time (e.g., a policy change or event), try training only on pre-period data, then predicting the post-period to form a counterfactual. Sketch your approach below.

In [ ]:
# Pseudocode template:
# 1) Identify a time column and a cutoff date.
# 2) Train your model on the pre-period only.
# 3) Predict into the post-period; compute (actual - predicted) as an effect proxy.
# 4) Summarize or plot the gap over time.
pass

## 8) Reflection (write 3–5 sentences)
- What did ML add beyond OLS here?  
- Where could overfitting or leakage occur?  
- How does this connect to Varian’s key messages about prediction vs. estimation?